# Intervalo de confianza de Wilson para SIRAP

Calcula el IC score de Wilson al 95 % para proporciones binarias del holdout SAVH. El tamaño muestral es el número de imágenes con referencia aprobada. Para video, cuadros o lecturas repetidas de una misma pasada no cuentan como ensayos independientes.


In [ ]:
from statistics import NormalDist

def wilson_interval(successes: int, trials: int, confidence_level: float = 0.95):
    """Devuelve el intervalo score de Wilson para una proporción binomial."""
    if isinstance(successes, bool) or isinstance(trials, bool):
        raise TypeError("successes y trials deben ser enteros")
    if not isinstance(successes, int) or not isinstance(trials, int):
        raise TypeError("successes y trials deben ser enteros")
    if trials <= 0 or not 0 <= successes <= trials:
        raise ValueError("Se requiere trials > 0 y 0 <= successes <= trials")
    if not 0 < confidence_level < 1:
        raise ValueError("confidence_level debe estar entre 0 y 1")
    p = successes / trials
    z = NormalDist().inv_cdf(0.5 + confidence_level / 2)
    z2 = z * z
    denom = 1 + z2 / trials
    center = (p + z2 / (2 * trials)) / denom
    half = z * (p * (1 - p) / trials + z2 / (4 * trials**2))**0.5 / denom
    return {"successes": successes, "trials": trials, "estimate": p,
            "confidence_level": confidence_level,
            "lower": max(0.0, center - half), "upper": min(1.0, center + half),
            "method": "wilson_score"}


In [ ]:
counts = {
    "OCR base: coincidencia exacta": (42, 57),
    "OCR afinado: coincidencia exacta": (35, 57),
    "OCR base: cobertura": (56, 57),
    "OCR afinado: cobertura": (56, 57),
}
for label, (successes, trials) in counts.items():
    ci = wilson_interval(successes, trials)
    print(f"{label}: {successes}/{trials} = {ci['estimate']:.1%}; "
          f"IC Wilson 95 % [{ci['lower']:.1%}, {ci['upper']:.1%}]")


## Lectura de los resultados

El OCR base obtuvo 42 aciertos en 57 imágenes: 73,7 %, con IC de Wilson 95 % de 61,0–83,4 %. El OCR afinado obtuvo 35/57: 61,4 %, con IC de 48,4–72,9 %.

El intervalo resume incertidumbre sobre una proporción binomial. No se aplica directamente a métricas continuas como CER, latencia o mAP. Tampoco se calcula sobre lecturas repetidas de video como si cada cuadro fuera independiente.

Fuente metodológica: NIST/SEMATECH e-Handbook, [Wilson confidence intervals](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm).